# 02 — Model Training

Full GBM pipeline with nested CV for 3d, 7d, 28d models.

In [1]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
import warnings; warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
from data_utils import load_csvs, clean, add_growth_features, add_bogue_features, build_feature_sets, BASE_FEATURES, OPT_FEATURES, avail_greedy
from model_utils import train_and_evaluate, plot_diagnostics, performance_table

CSV_DIR = r'C:\Users\artemis.01\Desktop\AI_RESISTANCE\AI_RESISTANCE\PRODUCAO\CSV_PI'

raw = load_csvs(CSV_DIR, 'cpii')
df  = clean(raw)
df  = add_growth_features(df)
df, derived = add_bogue_features(df)
feat, quim = build_feature_sets(df, BASE_FEATURES, OPT_FEATURES, derived)

TARGET = {'3d':'cs_3d', '7d':'cs_7d', '28d':'cs_28d'}
print(f'Dataset: {df.shape}')
print(f'QUIM ({len(quim)}): {quim}')
for h, cols in feat.items():
    n = df[cols + [TARGET[h]]].dropna().shape[0]
    print(f'  {h}: {n} samples  |  {len(cols)} features')

Dataset: (916, 23)
QUIM (13): ['na2o', 'fe2o3', 'cao', 'so3', 'blaine', 'sio2', 'pf', '#400', 'r.i', 'mgo', 'lsf', 'sm', 'blaine_so3']
  3d: 849 samples  |  13 features
  7d: 849 samples  |  14 features
  28d: 849 samples  |  16 features


## Train all horizons

In [2]:
results = {}
for h in ['3d', '7d', '28d']:
    target = TARGET[h]
    cols   = feat[h]
    df_h   = df[cols + [target]].dropna()
    X, y   = df_h[cols].values, df_h[target].values
    print(f'\n--- {h} ({len(df_h)} samples) ---')
    r = train_and_evaluate(X, y, baseline=True)
    r['feat'] = cols; r['X'] = X; r['y'] = y
    results[h] = r
    print(f'  GBM   R2={r["r2"].mean():.3f}  MAE={r["mae"].mean():.3f}')
    print(f'  Ridge R2={r["baseline"]["r2"].mean():.3f}  MAE={r["baseline"]["mae"].mean():.3f}')
    print(f'  Best params: {r["params"]}')


--- 3d (849 samples) ---


  GBM   R2=0.428  MAE=1.102
  Ridge R2=0.348  MAE=1.182
  Best params: {'gradientboostingregressor__learning_rate': 0.05, 'gradientboostingregressor__max_depth': 3, 'gradientboostingregressor__min_samples_leaf': 3, 'gradientboostingregressor__n_estimators': 300, 'gradientboostingregressor__subsample': 0.8}

--- 7d (849 samples) ---


  GBM   R2=0.777  MAE=0.700
  Ridge R2=0.785  MAE=0.679
  Best params: {'gradientboostingregressor__learning_rate': 0.05, 'gradientboostingregressor__max_depth': 3, 'gradientboostingregressor__min_samples_leaf': 3, 'gradientboostingregressor__n_estimators': 300, 'gradientboostingregressor__subsample': 0.8}

--- 28d (849 samples) ---


  GBM   R2=0.779  MAE=0.950
  Ridge R2=0.779  MAE=0.938
  Best params: {'gradientboostingregressor__learning_rate': 0.05, 'gradientboostingregressor__max_depth': 3, 'gradientboostingregressor__min_samples_leaf': 3, 'gradientboostingregressor__n_estimators': 300, 'gradientboostingregressor__subsample': 0.8}


## Performance table

In [3]:
print(f'{"Horizon":<8} {"GBM R²":>8} {"GBM MAE":>9} {"Ridge R²":>9} {"Ridge MAE":>10}')
print('-' * 50)
for h, r in results.items():
    bl = r['baseline']
    print(f'{h:<8} {r["r2"].mean():>8.3f} {r["mae"].mean():>9.3f} '
          f'{bl["r2"].mean():>9.3f} {bl["mae"].mean():>10.3f}')

Horizon    GBM R²   GBM MAE  Ridge R²  Ridge MAE
--------------------------------------------------
3d          0.428     1.102     0.348      1.182
7d          0.777     0.700     0.785      0.679
28d         0.779     0.950     0.779      0.938


## Diagnostic plots

In [4]:
fig = plot_diagnostics(results, derived, title='CP II-F v2',
                       save_path='../figures/diagnostico_cpiif_v2.png')
plt.close()